# 팀 실습 — 공개 정리본
교육 과정의 제공 골격을 바탕으로 수행한 실습입니다. 개인/팀 범위를 폴더로 구분했습니다.
공개 정리 시 환경 연결 셀·데이터 미리보기·내장 교안 이미지를 제외했습니다. 학습 설정과 모델 로직은 원본을 유지합니다.
개인 노트북의 평가 텍스트는 당시 저장 결과이며 나머지 실행 출력은 정리했습니다.


In [ ]:
import os


# **Step2. Multi-label 분류**



**미션 : 리뷰에 포함된 11가지 속성을 검출하기(Multi-label Multi-Classification)**

1) 목표 : 리뷰 내용에서, 속성에 해당하는 내용이 등장하는지 분류하는
모델 생성
2) 절차
    * 데이터 전처리 : 주어진 데이터를 목표에 맞는 형태로 변환
    * 모델링 : 다중 레이블 분류 파인 튜닝
        * 일반 파인 튜닝
        * LoRA 파인 튜닝

## 1.환경준비

### (1) 라이브러리 설치 및 로딩

In [ ]:
# 한글폰트 설치
import matplotlib as mpl
import matplotlib.font_manager as fm

%config InlineBackend.figure_format = 'retina'
!apt -qq -y install fonts-nanum

font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fontprop = fm.FontProperties(fname=font_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import torch

from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder

from peft import get_peft_model, LoraConfig, TaskType

import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

### (2) GPU 설정
* GPU 사용을 위한 설정

In [ ]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2.데이터 준비

### (1) 데이터 로딩
* data.csv 파일 로딩

In [ ]:
# 공개본: 환경 연결 셀은 생략. 데이터 경로는 DATA_PATH로 지정합니다.


### (2) Dataset 준비1

In [ ]:
# 레이블 리스트
labels = ['보습', '기능', '밀착', '가격', '향', '편의성', '기타', '외형', '포장', '품질', '피부']

* multi-hot label 만들기
    * multi-hot label 값이 문자열이 아닌 리스트여야 합니다.
* multi-hot label을 추가한 데이터셋을 data1 으로 저장합니다.

In [ ]:
data["aspect"] = data["aspect"].apply(lambda x: [x] if isinstance(x, str) else x) #문자열 -> 리스트 변환

def make_multilabel(aspect_list):
    return [1 if label in aspect_list else 0 for label in labels] # 각 aspect 에 대해 0과 1을 부여

data["label_vector"] = data["aspect"].apply(make_multilabel) # make_multilabel 함수를 적용

data_grouped = data.groupby("text")["aspect"].apply(sum).reset_index()
data_grouped["label_vector"] = data_grouped["aspect"].apply(make_multilabel)

data1 = data_grouped.copy()

data1.head()

* 검증 : 제대로 생성되었는지 다음 코드로 검증해 봅니다.

In [ ]:
review = "(장점 ) 1.쓰기편하고 제형이 묽어서 발림성 좋다. 2.백탁없다 (단점)"
display(data[data['text'].str.startswith(review)])
display(data1[data1['text'].str.startswith(review)])

* 전체 데이터에서 10000건을 샘플링 합니다.(파인튜닝 시간 절약을 위해서)

In [ ]:
data_sampled = data1.sample(n=10000, random_state=42)

### (3) Dataset 준비2
* train, val 분할 7:3
* 텐서 데이터셋

In [ ]:
# train, val 분할 7:3
train_df, val_df = train_test_split(
    data_sampled,
    test_size=0.3,
    random_state=42
)


In [ ]:
# df로 부터 텐서 데이터셋 만들기
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
train_dataset[0]


### (4) 토크나이징

* 다중 레이블 분류 문제에서 label은 float 타입이어야 함

In [ ]:
# 토크나이저 다운로드
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

In [ ]:
# 전처리 함수 정의 : label을 float 타입으로 변환 포함
def preprocess_function(examples):
    # 텍스트를 토크나이징
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    # 라벨을 float 타입으로 변환 (필수!)
    tokenized["labels"] = [list(map(float, x)) for x in examples["label_vector"]]

    # from datasets import Sequence, Value
    # train_ts = train_ts.cast_column("label", Sequence(Value("float32")))
    # val_ts   = val_ts.cast_column("label", Sequence(Value("float32")))

    return tokenized

# 전처리 적용
train_tokenized = train_dataset.map(preprocess_function, batched=True)
val_tokenized = val_dataset.map(preprocess_function, batched=True)

print(train_tokenized[0])


## 3.Fine-Tuning1
* 일반 파인 튜닝으로

### (1) 사전학습 모델 준비
* problem_type="multi_label_classification"

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    "klue/bert-base",
    num_labels=11,  # ✅ 반드시 11
    problem_type="multi_label_classification"
)

### (2) 학습 설정

* TrainArguments

In [ ]:

training_args = TrainingArguments(
    output_dir="./results_v2",
    eval_strategy="epoch",          # 매 epoch마다 평가
    save_strategy="epoch",
    learning_rate=5e-5,             # 일반보다 약간 높게
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.02,
    load_best_model_at_end=True,
    logging_dir='./logs_lora',
    logging_steps=10,
    report_to="tensorboard"
)



* Trainer

In [ ]:
# 학습 시 추가 함수 준비
from torch.utils.data import DataLoader
from transformers import default_data_collator

# # 평가함수(선택 사항)
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     probs = 1 / (1 + np.exp(-logits))  # sigmoid
#     preds = (probs > 0.5).astype(int)
#     return {
#         "weighted_f1": f1_score(labels, preds, average="weighted"),
#         "macro_f1": f1_score(labels, preds, average="macro")
#     }

# 사용자 정의 data_collator(필수 사항)
def float_label_collator(features):
    batch = default_data_collator(features)
    batch["labels"] = batch["labels"].float()  # ✅ 중요
    return batch

In [ ]:
# Trainer 설정 :

trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=float_label_collator,  # ✅ 필수
)

### (3) 학습
* 7000건 학습데이터에 대해서, 학습 시간이 10~13분 정도 소요 됩니다.

In [ ]:
trainer.train()


### (4) 모델 검증평가

In [ ]:
eval_results = trainer.evaluate()
print(f"\n📊 Evaluation results: {eval_results}")

* 예측값

In [ ]:
# 예측 수행
predictions = trainer.predict(val_tokenized)
logits = predictions.predictions
labels = predictions.label_ids

# sigmoid로 multi-label 확률화
probs = torch.sigmoid(torch.tensor(logits)).numpy()

# 0.5 기준으로 예측 이진화
y_pred = (probs >= 0.5).astype(int)
y_true = labels

print(f"\n✅ 예측 완료! (shape: {y_pred.shape})")

In [ ]:
# micro/macro 평균 기준으로 계산
precision = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
accuracy = accuracy_score(y_true, y_pred)

print(f"\n📈 모델 평가 요약")
print(f"Precision (micro): {precision:.4f}")
print(f"Recall (micro): {recall:.4f}")
print(f"F1 Score (micro): {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
# 네가 사용한 labels 리스트 재사용
labels_list = ['보습', '기능', '밀착', '가격', '향', '편의성', '기타', '외형', '포장', '품질', '피부']

# 각 속성별 평가
precisions, recalls, f1s, supports = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

# 그래프 시각화
x = np.arange(len(labels_list))
width = 0.25

plt.figure(figsize=(10,6))
plt.bar(x - width, precisions, width=width, label='Precision', color='cornflowerblue')
plt.bar(x, recalls, width=width, label='Recall', color='orange')
plt.bar(x + width, f1s, width=width, label='F1-score', color='seagreen')

plt.xticks(x, labels_list, rotation=45)
plt.ylim(0, 1.1)
plt.ylabel("Score")
plt.title("속성별 성능 비교 (Precision, Recall, F1)")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 각 속성별 점수 출력
for i, label in enumerate(labels_list):
    print(f"{label:<5} | Precision: {precisions[i]:.3f} | Recall: {recalls[i]:.3f} | F1: {f1s[i]:.3f} | Samples: {supports[i]}")

* 실제값

In [ ]:
# 예측 결과 확인용 (앞부분만 출력)
for i in range(5):
    print(f"\n리뷰 {i+1}:")
    print("📝 원문:", val_df.iloc[i]["text"])
    print("✅ 실제 라벨:", [label for j, label in enumerate(labels_list) if y_true[i][j] == 1])
    print("🤖 예측 라벨:", [label for j, label in enumerate(labels_list) if y_pred[i][j] == 1])

* 평가

In [ ]:
weighted_f1 = f1_score(labels, preds, average="weighted")
macro_f1 = f1_score(labels, preds, average="macro")

In [ ]:
print("\n🔹 모델 성능 평가 결과 🔹")
print(f"Weighted F1 Score: {weighted_f1:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")

In [ ]:
# 필요 시 세부 보고서
print("\n세부 분류 리포트:\n")
print(classification_report(labels, preds, digits=3))

In [ ]:
train_loss = []
eval_loss = []
epochs = []

for log in trainer.state.log_history:
    if "loss" in log:
        train_loss.append(log["loss"])
        epochs.append(log.get("epoch", len(epochs)))
    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8, 5))
plt.plot(epochs[:len(train_loss)], train_loss, label="Train Loss", marker='o')
plt.plot(epochs[:len(eval_loss)], eval_loss, label="Eval Loss", marker='s')
plt.title("Training vs Evaluation Loss (LoRA Fine-Tuning)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4.Fine-Tuning2
* LoRA 튜닝하기

### (1) 사전학습 모델 준비
* problem_type="multi_label_classification"

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
"klue/bert-base",
num_labels=11,
problem_type = "multi_label_classification"
)

### (2) LoRA 구성

In [ ]:
# LoRA 구성 정의
from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # Sequence Classification
    r=8,                         # 랭크(저차원 행렬 크기)
    lora_alpha=32,               # LoRA scaling factor
    lora_dropout=0.1,            # dropout 비율
    bias="none"                  # bias 업데이트 안 함
)


# LoRA 적용 (일부 레이어만 학습)
base_model = get_peft_model(base_model, lora_config)
base_model.print_trainable_parameters()  # 학습 가능한 파라미터만 출력


### (3) 학습 설정

* TrainArguments
    * 일반 파인 튜닝보다는 학습률을 조금 높게 설정

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_lora_v2",
    eval_strategy="epoch",          # 매 epoch마다 평가
    save_strategy="epoch",
    learning_rate=5e-5,             # 일반보다 약간 높게
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.02,
    load_best_model_at_end=True,
    logging_dir='./logs_lora',
    logging_steps=10,
    report_to="tensorboard"
)
print("✅ LoRA 학습 설정 완료")


* Trainer

In [ ]:
# 사용자 정의 data_collator 생
from torch.utils.data import DataLoader
from transformers import default_data_collator

def float_label_collator(features):
    batch = default_data_collator(features)
    batch.pop("num_items_in_batch", None)  # 🚫 Trainer 내부 키 제거
    batch["labels"] = batch["labels"].float()
    return batch

In [ ]:
# Trainer 설정
trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer = tokenizer,
    data_collator=float_label_collator,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)
print("✅ Trainer 구성 완료")


### (4) 학습

In [ ]:
trainer.train()

In [ ]:
torch.cuda.empty_cache()  # 사용 안 하는 GPU 메모리 정리

### (5) 모델 검증평가

In [ ]:
eval_results = trainer.evaluate()
print(f"\n📊 Evaluation results: {eval_results}")

In [ ]:
# 1️⃣ 검증셋 예측
predictions = trainer.predict(val_tokenized)
logits = predictions.predictions
labels = predictions.label_ids

# 2️⃣ sigmoid로 multi-label 확률화 후 이진화 (0.5 기준)
probs = torch.sigmoid(torch.tensor(logits)).numpy()
y_pred = (probs >= 0.5).astype(int)
y_true = labels

# 3️⃣ 각 라벨 이름 정의 (네가 사용한 labels 리스트와 동일하게)
label_names = ['보습', '기능', '밀착', '가격', '향', '편의성', '기타', '외형', '포장', '품질', '피부']

# 4️⃣ 각 속성별 precision, recall, f1 계산
precisions, recalls, f1s, supports = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

# 5️⃣ 시각화
x = np.arange(len(label_names))
width = 0.25

plt.figure(figsize=(10,6))
plt.bar(x - width, precisions, width=width, label='precision', color='cornflowerblue')
plt.bar(x, recalls, width=width, label='recall', color='orange')
plt.bar(x + width, f1s, width=width, label='f1', color='mediumseagreen')

plt.xticks(x, label_names, rotation=45)
plt.ylim(0, 1.2)
plt.ylabel('Score')
plt.xlabel('Label')
plt.title('속성별 성능 비교 (정밀도, 재현율, F1)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 6️⃣ 콘솔에 성능 출력
for i, label in enumerate(label_names):
    print(f"{label:<5} | Precision: {precisions[i]:.3f} | Recall: {recalls[i]:.3f} | F1: {f1s[i]:.3f} | Samples: {supports[i]}")